In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "00-foundation"))

import numpy as np
from Autograd import WhyyTorch as wt

In [2]:
# Just Handwritten Forward Pass 

x = [2.0,3.0]
w = [1.0,2.0]

y = []

hid_st = [0.0]
w_x = 1.0
w_h = 1.0
w_y = 2.0

for i in range(len(x)):
    h = np.tanh(w_h * hid_st[i] + w_x * x[i])
    hid_st.append(h)

y = hid_st[-1] * w_y
print(hid_st,y)


[0.0, np.float64(0.9640275800758169), np.float64(0.9992792861663325)] 1.998558572332665


In [3]:
import numpy as np

# Data
x = np.array([1.0, 2.0, 3.0])
target = 0.0

# Weights
Wx = 1.0
Wh = 0.5
Wy = 1.0


# Forward
h = [0.0]  

for i in range(len(x)):
    h_new = np.tanh(Wh * h[i] + Wx * x[i])
    h.append(h_new)

y = Wy * h[-1]

loss = 0.5 * (y - target) ** 2

print("Hidden states:", h)
print("Prediction:", y)
print("Loss:", loss)

# Backward Pass
dWh = 0.0
dWx = 0.0
dWy = 0.0

# Loss -> output
dy = y - target

# Output weight gradient
dWy = dy * h[-1]

# Gradient entering final hidden state
dh = dy * Wy

# Back through time
for i in reversed(range(len(x))):

    # Through tanh
    dz = dh * (1 - h[i + 1] ** 2)

    # Accumulate gradients
    dWh += dz * h[i]
    dWx += dz * x[i]

    # Send gradient to previous hidden state
    dh = dz * Wh


# Updation
learning_rate = 0.01

Wh -= learning_rate * dWh
Wx -= learning_rate * dWx
Wy -= learning_rate * dWy


print("\nGradients:")
print("dWh =", dWh)
print("dWx =", dWx)
print("dWy =", dWy)

print("\nUpdated weights:")
print("Wh =", Wh)
print("Wx =", Wx)
print("Wy =", Wy)

Hidden states: [0.0, np.float64(0.7615941559557649), np.float64(0.9830411011980433), np.float64(0.998146762128968)]
Prediction: 0.998146762128968
Loss: 0.4981484793742713

Gradients:
dWh = 0.0036808298119165245
dWx = 0.011225890146566915
dWy = 0.9962969587485426

Updated weights:
Wh = 0.49996319170188086
Wx = 0.9998877410985343
Wy = 0.9900370304125146


In [4]:
# Training Loop
# Data
x_data = [1.0, 2.0, 3.0]
target = wt(100.0, requires_grad=False)

# Weights
Wh = wt(0.5)
Wx = wt(1.0)
Wy = wt(1.0)
lr = 0.1
epochs = 10

for epoch in range(epochs):
    Wh.zero_grad()
    Wx.zero_grad()
    Wy.zero_grad()

    h_prev = wt(0.0, requires_grad=False)
    h_states = [h_prev]
    # Forward pass
    for i in range(len(x_data)):
        xi = wt(x_data[i], requires_grad=False)
        h_new = (Wh * h_states[-1] + Wx * xi).tanh()
        h_states.append(h_new)

    # Output
    y = Wy * h_states[-1]
    # Loss
    loss = 0.5 * (y - target) ** 2

    # Backward pass
    loss.backward()

    # Update weights
    Wh.data -= lr * Wh.grad
    Wx.data -= lr * Wx.grad
    Wy.data -= lr * Wy.grad

    print(f"epoch {epoch}: loss={loss.data}, Wh={Wh.data}, Wx={Wx.data}, Wy={Wy.data}")

epoch 0: loss=4900.68359375, Wh=0.5365081429481506, Wx=1.111343502998352, Wy=10.88183879852295
epoch 1: loss=3971.875, Wh=0.7065134048461914, Wx=1.6259874105453491, Wy=19.786785125732422
epoch 2: loss=3217.125, Wh=0.7154650688171387, Wx=1.6528571844100952, Wy=27.807937622070312
epoch 3: loss=2605.894287109375, Wh=0.7249442338943481, Wx=1.6813081502914429, Wy=35.02703857421875
epoch 4: loss=2110.787353515625, Wh=0.7338176965713501, Wx=1.7079393863677979, Wy=41.52427673339844
epoch 5: loss=1709.7447509765625, Wh=0.7417505979537964, Wx=1.7317465543746948, Wy=47.371822357177734
epoch 6: loss=1384.897216796875, Wh=0.7487065196037292, Wx=1.7526206970214844, Wy=52.6346321105957
epoch 7: loss=1121.7694091796875, Wh=0.7547705173492432, Wx=1.7708176374435425, Wy=57.37117385864258
epoch 8: loss=908.6345825195312, Wh=0.7600193023681641, Wx=1.78656804561615, Wy=61.634071350097656
epoch 9: loss=735.9952392578125, Wh=0.7646148800849915, Wx=1.8003580570220947, Wy=65.47068786621094
